## torch forward

In [92]:
%load_ext autoreload
%autoreload 2

import sys
    
sys.path.append('/home/lishengping/projects/dreamily-v3.5-deploy/app')

import torch
import sentencepiece as spm

from configuration_dcformer import ModelConfig
from modeling_dcformer import DCFormer

TOKENIZER_PATH = '/home/lishengping/tokenizer/spm_model_70000vocab_55G_bpe_character_coverage0.99999.extended_special.model'
tokenizer = spm.SentencePieceProcessor(model_file=TOKENIZER_PATH)

torch_config = ModelConfig()
torch_config.torch_dtype = torch.bfloat16

model = DCFormer(torch_config)
torch_params = torch.load('torch_params.bin') # torch.save之后的参数
IncompatibleKeys = model.load_state_dict(torch_params, strict=False)
for m in IncompatibleKeys.missing_keys:
    if 'layers.0' in m:
        print(m)
        
model.eval()
# model.half()
model.bfloat16()

with torch.device(model.device):
    model.setup_caches(max_batch_size=1, set_kv_cache=True)

layers.0.attention.dyn_w_proj.dw_m
layers.0.attention.dyn_w_proj.qkw_m
layers.0.attention.dyn_w_proj.qkw_bias


In [87]:
import os
import time
import argparse
import socket
import random
from collections import defaultdict

os.environ["JAX_PLATFORMS"] = "cpu"

import tensorflow as tf
import jax
import numpy as np


def _parse_function(example_proto):
    feature_desc = {key: tf.io.VarLenFeature(tf.int64) for key in task_features}
    example = tf.io.parse_single_example(example_proto, feature_desc)
    for name in list(example.keys()):
        t = example[name]
        if t.dtype == tf.int64:
            t = tf.cast(t, dtype=tf.int32)
        example[name] = tf.sparse.to_dense(t, default_value=0)[: seq_len]
        print(f'example[name]: {example[name]}')
    return example

task_features = {'input_ids': None}
train_seed = 1234
num_infeed_hosts = 1
shuffle_buffer_size = None
pad_id = 0
batch_size = 1
seq_len = 4097

p = 'gs://newproject-1-jax_llm_data_europe-west4/xiaomeng/v3.5mini/unigram_tfids0506/validation/R000.000000'
fname = [p]
tf.random.set_seed(train_seed)
ds = tf.data.Dataset.from_tensor_slices(fname)
ds = ds.apply(tf.data.TFRecordDataset)
ds = ds.shard(num_infeed_hosts, 0)
ds = ds.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
if shuffle_buffer_size is not None:
    ds = ds.shuffle(buffer_size=shuffle_buffer_size)
padded_shapes = {key: seq_len for key in task_features}
padding_values = {key: pad_id for key in task_features}
ds = ds.padded_batch(
    batch_size=np.prod(batch_size),
    padded_shapes=padded_shapes,
    padding_values=padding_values,
    drop_remainder=True,
)
ds_iter = ds.as_numpy_iterator()

example[name]: Tensor("strided_slice:0", shape=(None,), dtype=int32)


In [95]:
length = 250
torch_bf16_losses = []
for i in range(20):
    a = next(ds_iter)
    batch_indexes = torch.tensor([0])
    inputs = torch.from_numpy(a['input_ids'][:, :length]).long()
    labels = torch.from_numpy(a['input_ids'][:, 1:length+1]).long()
    input_pos = torch.arange(length).unsqueeze(0)
    
    with torch.no_grad():
        output = model.forward(inputs, input_pos, batch_indexes=batch_indexes)
    
    loss_fn = torch.nn.CrossEntropyLoss(reduction='none')
    
    loss = loss_fn(output.logits.view(-1, 70000).float(), labels.view(-1))
    loss = round(loss.mean().item(), 4)
    print(loss)
    torch_bf16_losses.append(loss)

## jax forward

In [ ]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import yaml

sys.path.append('/home/lishengping/projects/maxtext/MaxText')
os.environ['HARDWARE'] = 'cpu'

import pyconfig
from layers import models
import max_utils
import jax
import orbax
import jax.numpy as jnp
from jax.sharding import Mesh
from flax.traverse_util import flatten_dict, unflatten_dict
from flax import linen as nn

# 配置文件需要更改的几个地方：
# base_output_directory: 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL_32k_0530'
# query_chunk_size = None # 如果传了这个参数，forward需要是query_chunk_size的整数倍
# attention = 'dot_product_chunk'
    # exp_class set your model class
# per_device_batch_size = 1 # 可以根据测试的batch size定，设小一点主要是为了节省显存
# max_target_length = 256 # 可以根据测试的长度定，设小一点主要是为了节省显存
# zero_loss = True
# run_name = 'test' # 因为base.yml默认为空，必须写一个
# scan_layers = False # 因为转模型的时候转的是scan_layers=False
run_name = 'test'
os.makedirs(run_name, exist_ok=True) # 因为如果不存在会报错
config_name = '/home/lishengping/projects/maxtext/MaxText/configs/base.yml'
argv = [None, config_name]
config = pyconfig.initialize(argv)

In [ ]:
import torch
import numpy as np
import jax.numpy as jnp
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


shapedtype = {l.split(' (')[0].strip(): eval('(' + l.split(' (')[1].strip()) for l in open('mini_params.txt', 'r') if l.strip()}

# load
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
axes = [1] * len(mesh_axes)
axes[2] = 4
devices = np.asarray(jax.devices()).reshape(axes)
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = jnp.bfloat16 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
for k, shape in shapedtype.items():
    if not isinstance(k, tuple):
        k = tuple(k.split('/'))
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)    
    
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)['params']

checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL_32k_0531/checkpoints/0/items'

ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args}
)

In [ ]:
def model_init(model, config, key):
  input_shape = (config.global_batch_size_to_load, config.max_target_length)
  params = model.init(
      {"params": key, "dropout": key, "aqt": key},
      jnp.ones(input_shape, dtype=jnp.int32),
      jnp.ones(input_shape, dtype=jnp.int32),
  )
  return params

quant = None
devices_array = max_utils.create_device_mesh(config)
mesh = Mesh(devices_array, config.mesh_axes)
Transformer = models.Transformer
jax_model = Transformer(config, mesh, quant=quant)

is_train = False
rng1, aqt_rng = jax.random.split(jax.random.key(9876))


In [ ]:
jax_losses = []
for i in range(20):
    a = next(ds_iter)
    batch_indexes = torch.tensor([0])
    inputs = torch.from_numpy(a['input_ids'][:, :length]).long()
    labels = torch.from_numpy(a['input_ids'][:, 1:length+1]).long()
    input_pos = torch.arange(length).unsqueeze(0)

    batch_size = 1
    input_ids = jnp.array(inputs).reshape(batch_size, -1)
    data = {}
    data['inputs'] = input_ids
    data["inputs_position"] = jnp.arange(data['inputs'].shape[1]).reshape(batch_size, -1)
    data["inputs_segmentation"] = jnp.ones_like(data['inputs'])
    data["targets"] = jnp.array(labels).reshape(batch_size, -1)
    
    
    params = restored['params']['params']
    jax_logits, intermediate_outputs = jax_model.apply(
          {'params': params},
          data["inputs"],
          data["inputs_position"],
          decoder_segment_ids=data["inputs_segmentation"],
          enable_dropout=config.enable_dropout if is_train else False,
          rngs={"dropout": rng1, "params": aqt_rng},
          mutable="intermediates",
      )
    one_hot_targets = jax.nn.one_hot(data["targets"], config.vocab_size)
    jax_loss, _ = max_utils.cross_entropy_with_logits(jax_logits, one_hot_targets, 0.0)
    loss = round(jax_loss.mean().item(), 4)
    # print(f'jax mean loss: {jax_loss.mean()} shape: {jax_loss.shape}')
    print(loss)
    jax_losses.append(loss)
    

In [ ]:
# key = 'decoder/layers_0/sub_0/[lsp]inputs'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]attn_inputs_q'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]attn_inputs_k'
# # key = 'decoder/layers_0/sub_0/self_attention/[lsp]attn_query'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]attn_key'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]shift_key'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]norm_key'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]norm_query'

# key = 'decoder/layers_0/sub_0/self_attention/[lsp]rotary_key'
# key = 'decoder/layers_0/sub_0/self_attention/[lsp]rotary_query'

# key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]attn_weights'
# key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]pre_proj_attn_weights'
# # key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]softmax_before_attn_weights'

# # key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]pre_qw1'
# # key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]post_qw1'

# key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]softmax_after_probs'
# # key = 'decoder/layers_0/sub_0/self_attention/attention_op/QChunk_0/[lsp]post_proj_probs'
# # key = 'decoder/layers_1/sub_0/self_attention/attention_op/q_dyn_w_proj/[lsp]dd'

# # key = 'decoder/layers_1/sub_0/self_attention/[lsp]attn_out'


# debug_outputs[key][0].sum(), debug_outputs[key][0].shape, debug_outputs[key][0]